# 6章 サンプル（ライブラリなし版） ― 固有値計算を自分で書く（Jacobi法）

06-2pca_numpy.ipynb では `np.linalg.eigh` に任せていた固有値・固有ベクトルの計算を、
自分で実装して中身を開けてみる。使うアルゴリズムは、06-1Reduce.htmlのJavaScriptに実装されている
**古典的Jacobi法**（対称行列を、非対角成分が大きい場所から順に回転させて消していく反復計算）を、
そのままPythonに移植したもの。データ・標準化・共分散行列の作り方は06-2と完全に同じにしてあるので、
最後の「射影後の座標（scores）」がライブラリ版とぴったり一致することを確認できる。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Colabで日本語が文字化けしないようにするためのおまじない
try:
    import japanize_matplotlib
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'japanize-matplotlib'])
    import japanize_matplotlib

# 06-1Reduce.html の defaultData、06-2pca_numpy.ipynb と完全に同じデータ
car_names = ['軽自動車', 'コンパクト', 'セダン', 'SUV', 'スポーツカー']
car_colors = ['#C1502E', '#4A6D7C', '#8A7A3F', '#6B4E8E', '#3E7A5B']
spec_names = ['排気量(L)', '馬力(PS)', '重量(kg)', '燃費(km/L)', '価格(万円)']

X = np.array([
    [0.66,  52,  780, 21, 150],   # 軽自動車
    [1.5,  110, 1050, 18, 220],   # コンパクト
    [2.0,  160, 1400, 15, 320],   # セダン
    [2.5,  220, 1800, 11, 420],   # SUV
    [3.5,  350, 1500,  8, 800],   # スポーツカー
], dtype=float)


## ①②標準化と共分散行列（06-2と同じ）


In [ ]:
# ①②標準化と共分散行列
mean = X.mean(axis=0)
std = X.std(axis=0, ddof=0)
Z = (X - mean) / std

n = X.shape[0]
C = (Z.T @ Z) / (n - 1)

print("共分散行列 C:\n", np.round(C, 4))


## ③固有値問題を自分で解く ― 古典的Jacobi法

対称行列は、適切な回転を繰り返し施すことで、いずれ対角行列（非対角成分がすべて0）に変形できる。
そうなったときの対角成分が固有値、そこに至るまでにかけた回転を全部掛け合わせたものが固有ベクトルになる。

手順:
1. 今、絶対値が一番大きい非対角成分 `a[p][q]` を探す
2. その成分がちょうど0になるように、(p, q)の2次元だけを回転させる角度 $\theta$ を計算する
3. 実際にその回転を行列全体に適用する（対角成分・他の非対角成分も巻き込まれて変化する）
4. これを、非対角成分が十分小さくなるまで繰り返す

回転角の公式は、$a'_{pq}=0$ となる条件から導ける。

$$\theta = \frac{1}{2}\arctan2\bigl(2\,a_{pq},\ a_{qq}-a_{pp}\bigr)$$

（$\arctan2$の第2引数の符号を逆にすると、回転はできるが目的の成分が0にならず、
　最終的な固有値が正しく求まらない。実際に試して確認したので、注意点として残しておく）


In [ ]:
# ③固有値問題を自分で解く（Jacobi法）
def jacobi_eigen(A, max_iter=200, tol=1e-12):
    a = A.copy().astype(float)
    n = a.shape[0]
    v = np.eye(n)   # 回転を掛け合わせていく行列。最終的にこれが固有ベクトルの集まりになる

    for _ in range(max_iter):
        # ① 絶対値が最大の非対角成分(p, q)を探す
        off = 0.0
        p, q = 0, 1
        for i in range(n):
            for j in range(i + 1, n):
                if abs(a[i, j]) > off:
                    off = abs(a[i, j])
                    p, q = i, j

        if off < tol:      # 十分小さくなったら終了
            break

        # ② a[p][q]をちょうど0にする回転角を求める
        if abs(a[p, p] - a[q, q]) < 1e-15:
            theta = np.pi / 4 * (1 if a[p, q] >= 0 else -1)
        else:
            theta = 0.5 * np.arctan2(2 * a[p, q], a[q, q] - a[p, p])
        c, s = np.cos(theta), np.sin(theta)

        # ③ 回転を行列全体に適用する
        app = c*c*a[p, p] - 2*s*c*a[p, q] + s*s*a[q, q]
        aqq = s*s*a[p, p] + 2*s*c*a[p, q] + c*c*a[q, q]
        for k in range(n):
            if k != p and k != q:
                akp, akq = a[k, p], a[k, q]
                a[k, p] = a[p, k] = c*akp - s*akq
                a[k, q] = a[q, k] = s*akp + c*akq
        a[p, p], a[q, q] = app, aqq
        a[p, q] = a[q, p] = 0.0

        for k in range(n):
            vkp, vkq = v[k, p], v[k, q]
            v[k, p] = c*vkp - s*vkq
            v[k, q] = s*vkp + c*vkq

    eigenvalues = np.diag(a).copy()
    eigenvectors = v.copy()   # 列ベクトルがそれぞれの固有ベクトル
    return eigenvalues, eigenvectors


values, vectors = jacobi_eigen(C)

order = np.argsort(values)[::-1]
values = values[order]
vectors = vectors[:, order]

# 符号をそろえる（06-2と同じ規則。html版のrunPCA()と同じ）
for k in range(vectors.shape[1]):
    idx = np.argmax(np.abs(vectors[:, k]))
    if vectors[idx, k] < 0:
        vectors[:, k] *= -1

ratio = np.maximum(values, 0) / np.maximum(values, 0).sum()

print("固有値(降順):", np.round(values, 4))
print("寄与率(%):", np.round(ratio * 100, 2))


## ④上位2本を採用して、5次元→2次元に射影する（06-2と同じ）


In [ ]:
# ④上位2本を採用して射影（スコア）
W = vectors[:, :2]
scores = Z @ W
loadings = W * np.sqrt(np.maximum(values[:2], 0))

print("W (5x2):\n", np.round(W, 3))
print("\nscores (5x2):\n", np.round(scores, 3))


## ④の検算：06-2pca_numpy.ipynb（`eigh`版）との一致確認

固有値計算のアルゴリズムはまったく別物（Jacobi法 vs LAPACKのeigh）だが、
最終的な座標（scores）が一致するはずである。06-2のscoresをここに貼って、差を確認する。


In [ ]:
# 06-2pca_numpy.ipynb の scores（eigh版）をそのまま貼り付けたもの
scores_lib = np.array([
    [ 2.915, -0.426],
    [ 1.508, -0.104],
    [ 0.155,  0.391],
    [-1.398,  0.997],
    [-3.179, -0.859],
])

print("Jacobi版とeigh版の差の最大値:", np.max(np.abs(np.abs(scores) - np.abs(scores_lib))))
print("(符号は固有ベクトルの向きの取り方で反転することがあるため、絶対値で比較している)")


## ⑤可視化（06-2pca_numpy.ipynbとまったく同じ見せ方）


In [ ]:
# ⑤可視化
fig, ax = plt.subplots(figsize=(6, 6))

arrow_scale = 3.0
for j, name in enumerate(spec_names):
    ax.annotate('', xy=(loadings[j, 0]*arrow_scale, loadings[j, 1]*arrow_scale), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='#7d746a', lw=1.2))
    ax.text(loadings[j, 0]*arrow_scale*1.1, loadings[j, 1]*arrow_scale*1.1,
            name.split('(')[0], color='#7d746a', fontsize=9)

for i, name in enumerate(car_names):
    ax.scatter(scores[i, 0], scores[i, 1], color=car_colors[i], s=70, zorder=3)
    ax.annotate(name, (scores[i, 0], scores[i, 1]), xytext=(6, 6),
                textcoords='offset points', fontsize=10)

ax.axhline(0, color='#D8D3C7', lw=1)
ax.axvline(0, color='#D8D3C7', lw=1)
ax.set_xlabel(f'PC1  ({ratio[0]*100:.1f}%)')
ax.set_ylabel(f'PC2  ({ratio[1]*100:.1f}%)')
ax.set_title('PCA: PC1 × PC2（バイプロット） ― ライブラリなし版(Jacobi法)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()


## 参考：圧縮前の元データ一覧

2次元に落とし込む前に、元々どれだけの情報（5諸元 × 5台）があったのかを、あらためて一覧で確認する。

In [ ]:
import pandas as pd

pd.DataFrame(X, index=car_names, columns=spec_names)
